# Testing YOLO on test data

In [2]:
import os
from pathlib import Path
from sahi.slicing import slice_image

# Paths relative to Test/src/test.ipynb
RAW_BASE_DIR = Path("../res/Uncompressed")
SPECIES_SUBFOLDERS = ["Lactobacillus.delbrueckii", "Lactobacillus.salivarius"]

OUTPUT_TILES_DIR = Path("../dataset/images/train")
OUTPUT_TILES_DIR.mkdir(parents=True, exist_ok=True)

# Gather all .tif files from both species folders
tif_files = []
for species in SPECIES_SUBFOLDERS:
    species_dir = RAW_BASE_DIR / species
    found = list(species_dir.glob("*.tif")) + list(species_dir.glob("*.tiff")) + \
            list(species_dir.glob("*.TIF")) + list(species_dir.glob("*.TIFF"))
    print(f"Found {len(found)} .tif files in {species}")
    tif_files.extend(found)

print(f"\nTotal raw images to slice: {len(tif_files)}")

# Slice all .tif slides into 640x640 tiles
for img_path in tif_files:
    # Include species name in tile output filename to prevent naming collisions
    out_prefix = f"{img_path.parent.name}_{img_path.stem}"
    
    slice_image(
        image=str(img_path),
        output_dir=str(OUTPUT_TILES_DIR),
        output_file_name=out_prefix,
        slice_height=640,
        slice_width=640,
        overlap_height_ratio=0.2, # 20% overlap
        overlap_width_ratio=0.2,
        verbose=False
    )

print(f"\nSlicing complete! Tiles generated at: {OUTPUT_TILES_DIR.resolve()}")

Found 40 .tif files in Lactobacillus.delbrueckii
Found 40 .tif files in Lactobacillus.salivarius

Total raw images to slice: 80

Slicing complete! Tiles generated at: D:\proj\Liquid-Food-Analyzer\Test\dataset\images\train


In [3]:
import yaml
from pathlib import Path

DATASET_DIR = Path("../dataset").resolve()

data_config = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val': 'images/train',  # Point to images/val when you add a validation split
    'names': {
        0: 'Lactobacillus_delbrueckii',
        1: 'Lactobacillus_salivarius'
    }
}

yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print(f"Created data.yaml at: {yaml_path}")

Created data.yaml at: D:\proj\Liquid-Food-Analyzer\Test\dataset\data.yaml


In [4]:
import cv2
import numpy as np
from pathlib import Path

# Paths relative to Test/src/test.ipynb
TILED_IMAGES_DIR = Path("../dataset/images/train").resolve()
TILED_LABELS_DIR = Path("../dataset/labels/train").resolve()
TILED_LABELS_DIR.mkdir(parents=True, exist_ok=True)

image_files = list(TILED_IMAGES_DIR.glob("*.png")) + list(TILED_IMAGES_DIR.glob("*.jpg"))
print(f"Generating labels for {len(image_files)} image tiles...")

generated_count = 0

for img_path in image_files:
    # Determine class ID from filename (0 = delbrueckii, 1 = salivarius)
    class_id = 0 if "delbrueckii" in img_path.name.lower() else 1
    
    # Read tile image
    img = cv2.imread(str(img_path))
    if img is None:
        continue
        
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Adaptive thresholding to separate bacteria from background
    # (Adjust block size 15 and C constant 3 if needed based on lighting)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.adaptiveThreshold(
        blur, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY_INV, 15, 3
    )

    # Find bacterial contours (polygons)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    label_lines = []
    h, w = gray.shape

    for cnt in contours:
        # Filter out noise (ignore tiny specks under 10 pixels area)
        area = cv2.contourArea(cnt)
        if area < 10 or area > 5000:
            continue

        # Reduce contour complexity to clean polygon points
        epsilon = 0.01 * cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, epsilon, True)

        # Minimum 3 points required for a valid polygon mask
        if len(approx) >= 3:
            # Flatten and normalize coordinates between 0.0 and 1.0
            points = approx.reshape(-1, 2)
            norm_points = []
            for px, py in points:
                norm_points.append(f"{px / w:.5f}")
                norm_points.append(f"{py / h:.5f}")

            # Format: <class_id> x1 y1 x2 y2 ... xn yn
            line = f"{class_id} " + " ".join(norm_points)
            label_lines.append(line)

    # Write corresponding .txt file
    txt_path = TILED_LABELS_DIR / f"{img_path.stem}.txt"
    with open(txt_path, "w") as f:
        f.write("\n".join(label_lines))
        
    generated_count += 1

print(f"Done! Created {generated_count} label files in: {TILED_LABELS_DIR}")

Generating labels for 480 image tiles...
Done! Created 480 label files in: D:\proj\Liquid-Food-Analyzer\Test\dataset\labels\train


In [5]:
from pathlib import Path
import torch
from ultralytics import YOLO

DATASET_DIR = Path("../dataset").resolve()
yaml_path = str(DATASET_DIR / "data.yaml")

# Verify label count before training
label_files = list((DATASET_DIR / "labels/train").glob("*.txt"))
print(f"Found {len(label_files)} label files ready for training.")

# Load model
model = YOLO('yolo26n-seg.pt')

results = model.train(
    data=yaml_path,
    epochs=5,
    imgsz=640,
    batch=4,
    device=0,
    workers=2,
    cache=False,
    name='yolo_lactobacillus_gpu'
)

Found 480 label files ready for training.
Ultralytics 8.4.117  Python-3.12.9 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\proj\Liquid-Food-Analyzer\Test\dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, mom

In [7]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nModel exists:")
print(Path("runs/segment/yolo_lactobacillus/weights/best.pt").exists())

print("\nCPU model exists:")
print(Path("runs/segment/yolo_lactobacillus_cpu/weights/best.pt").exists())

print("\nAvailable TIFF files:")
for p in Path("../res/Uncompressed/Lactobacillus.delbrueckii").glob("*.tif"):
    print(p)

Current working directory:
d:\proj\Liquid-Food-Analyzer\Test\src

Model exists:
False

CPU model exists:
False

Available TIFF files:
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0001.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0002.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0003.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0004.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0005.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0006.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0007.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0008.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0009.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacillus.delbrueckii_0010.tif
..\res\Uncompressed\Lactobacillus.delbrueckii\Lactobacil

In [13]:
from pathlib import Path

cwd = Path.cwd()

print("CWD:")
print(cwd)

print("\nAll best.pt files under CWD:")

for p in cwd.rglob("best.pt"):
    print(p.resolve())

CWD:
d:\proj\Liquid-Food-Analyzer\Test\src

All best.pt files under CWD:
D:\proj\Liquid-Food-Analyzer\Test\src\runs\segment\yolo_lactobacillus_gpu\weights\best.pt


In [3]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.delbrueckii/Lactobacillus.delbrueckii_0001.tif" # Adjust filename as needed

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.delbrueckii_0001.tif ---
L. delbrueckii: 2109
L. salivarius:  1
Total Bacteria: 2110


In [6]:


from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.salivarius/Lactobacillus.salivarius_0001.tif"

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.salivarius_0001.tif ---
L. delbrueckii: 8
L. salivarius:  456
Total Bacteria: 464


In [2]:


from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.casei_0001.tif"

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.casei_0001.tif ---
L. delbrueckii: 422
L. salivarius:  313
Total Bacteria: 735


In [4]:


from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.delbrueckii/Lactobacillus.delbrueckii_0003.tif"

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.delbrueckii_0003.tif ---
L. delbrueckii: 594
L. salivarius:  1
Total Bacteria: 595


In [5]:


from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.delbrueckii/Lactobacillus.delbrueckii_0004.tif"

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.delbrueckii_0004.tif ---
L. delbrueckii: 1994
L. salivarius:  0
Total Bacteria: 1994


In [6]:


from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path

# Load fine-tuned model
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path="runs/segment/yolo_lactobacillus_gpu/weights/best.pt",
    confidence_threshold=0.25,
    device="cuda:0"
)

# Path to a sample .tif slide
sample_tif = "../res/Uncompressed/Lactobacillus.delbrueckii/Lactobacillus.delbrueckii_0016.tif"

result = get_sliced_prediction(
    sample_tif,
    detection_model,
    slice_height=640,
    slice_width=640,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    perform_standard_pred=False
)

# Save output visualization image
result.export_visuals(export_dir="../results/", file_name="counted_tif_slide")

# Tally bacteria variants
preds = result.object_prediction_list
delbrueckii_count = sum(1 for p in preds if p.category.id == 0)
salivarius_count  = sum(1 for p in preds if p.category.id == 1)

print(f"--- Bacteria Tally for {Path(sample_tif).name} ---")
print(f"L. delbrueckii: {delbrueckii_count}")
print(f"L. salivarius:  {salivarius_count}")
print(f"Total Bacteria: {len(preds)}")

Performing prediction on 12 slices.
--- Bacteria Tally for Lactobacillus.delbrueckii_0016.tif ---
L. delbrueckii: 2350
L. salivarius:  0
Total Bacteria: 2350
